# Proyecto 3 - Modelo final

El presente cuaderno guarda los resultados del mejor modelo XGBoost con datos limpios originales (no enriquecidos) en varios objetos .joblib para la posterior configuración de la aplicación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# 1. Leemos el archivo CSV limpio usando la ruta real de tu Google Drive
ruta_drive = '/content/drive/MyDrive/SP1668 Tecnicas computacionales ML/Proyecto 2/df_limpio.csv'
df_limpio = pd.read_csv(ruta_drive)

# 2. Comprobamos el tamaño del dataset (filas, columnas)
print(f"El dataset contiene {df_limpio.shape[0]} filas y {df_limpio.shape[1]} columnas.\n")


El dataset contiene 4656 filas y 18 columnas.



In [ ]:
df_limpio.head()

,age,gender,ethnicity,income_level,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,sleep_hours_per_day,screen_time_hours_per_day,family_history_diabetes,hypertension_history,cardiovascular_history,bmi,heart_rate,cholesterol_total,triglycerides,insulin_level,diagnosed_diabetes
0,73,Male,White,Lower-Middle,Never,0,107,6.1,8.5,0,0,0,23.6,82,224,69,5.90,0
1,34,Male,White,Upper-Middle,Never,1,234,7.6,5.7,0,1,0,22.7,77,168,182,2.75,0
2,57,Female,Black,Middle,Never,2,234,8.1,4.7,0,0,1,30.0,64,167,210,11.35,0
3,59,Female,Hispanic,High,Never,2,83,6.6,10.9,0,1,0,24.5,60,231,123,9.41,0
4,62,Male,Hispanic,Middle,Current,0,125,5.8,1.1,0,0,0,26.8,76,205,165,7.03,1


# Mejor modelo


In [ ]:
from sklearn.model_selection import train_test_split

# 1. Separamos las variables predictoras (X) de la variable respuesta (y)
# Suponiendo que 'diagnosed_diabetes' es tu target
X_final = df_limpio.drop(columns=['diagnosed_diabetes'])
y_final = df_limpio['diagnosed_diabetes']

# 2. Dividimos los datos
# Usualmente se usa un 70/30 o 80/20 para tesis académicas
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.20,      # 20% para probar el modelo
    stratify=y_final,    # Mantiene la proporción de diabéticos en ambos
    random_state=42      # Permite que tus resultados sean replicables
)

# 3. Verificamos las dimensiones con .shape
print(f"Dimensiones de Entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones de Prueba (X_test): {X_test.shape}")

# 4. Verificamos que la estratificación funcionó
print("\nProporción de Diabetes en Entrenamiento:")
print(y_train.value_counts(normalize=True).round(4))
print("\nProporción de Diabetes en Prueba:")
print(y_test.value_counts(normalize=True).round(4))

Dimensiones de Entrenamiento (X_train): (3724, 17)
Dimensiones de Prueba (X_test): (932, 17)

Proporción de Diabetes en Entrenamiento:
diagnosed_diabetes
1    0.6015
0    0.3985
Name: proportion, dtype: float64

Proporción de Diabetes en Prueba:
diagnosed_diabetes
1    0.6009
0    0.3991
Name: proportion, dtype: float64




*   Aplicación técnicas de encoding



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import RobustScaler, LabelEncoder, OneHotEncoder

# 1. PREPARACIÓN DEL DATASET COMPLETO (Sin referencias a age_cod)

# A. Label Encoding para income_level (Ordinal)
le_income = LabelEncoder()
X_train['income_level'] = le_income.fit_transform(X_train['income_level'])
# IMPORTANTE: En el test set solo aplicamos .transform() para evitar fuga de datos
X_test['income_level'] = le_income.transform(X_test['income_level'])

# B. One-Hot Encoding para las variables nominales usando Scikit-Learn
nominales = ['gender', 'ethnicity', 'family_history_diabetes',
             'hypertension_history', 'cardiovascular_history', 'smoking_status']

# Filtrar solo las columnas que realmente existan en X_train
nominales_presentes = [c for c in nominales if c in X_train.columns]

# Inicializamos el codificador (drop='first' equivale a drop_first=True)
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error')
ohe.set_output(transform="pandas")

# Ajustamos y transformamos las columnas nominales en el Train set
X_train_ohe = ohe.fit_transform(X_train[nominales_presentes])

# Transformamos las columnas nominales en el Test set usando el encoder entrenado
X_test_ohe = ohe.transform(X_test[nominales_presentes])

# Reconstruimos los conjuntos eliminando las columnas de texto originales y concatenando las codificadas
X_train = pd.concat([X_train.drop(columns=nominales_presentes), X_train_ohe], axis=1)
X_test = pd.concat([X_test.drop(columns=nominales_presentes), X_test_ohe], axis=1)

# C. ALINEACIÓN DE COLUMNAS (Garantía para XGBoost y Random Forest)
# Nota: Aunque el OneHotEncoder asegura las mismas columnas, el reindex garantiza que
# el ORDEN de absolutamente todas las variables (incluyendo numéricas u ordinales) sea idéntico.
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Verificación de control
print(f"Columnas en X_train: {X_train.shape[1]} | Columnas en X_test: {X_test.shape[1]}")
if list(X_train.columns) == list(X_test.columns):
    print("¡Éxito! Las matrices de diseño están perfectamente alineadas.")
else:
    print("Atención: Hay un desalineamiento en las columnas.")

Columnas en X_train: 21 | Columnas en X_test: 21
¡Éxito! Las matrices de diseño están perfectamente alineadas.


In [ ]:
# Verificación de dimensiones del flujo de trabajo
print("--- Dimensiones del Dataset ---")
print(f"Muestra original (limpia): {df_limpio.shape}") # Usamos df_limpio que es el resultado post-outliers
print("-" * 30)
print(f"X_train (Entrenamiento):    {X_train.shape}")
print(f"y_train (Target Entrenam.): {y_train.shape}")
print("-" * 30)
print(f"X_test  (Prueba):           {X_test.shape}")
print(f"y_test  (Target Prueba):    {y_test.shape}")

# Verificación de consistencia (Opcional pero recomendado)
if X_train.shape[0] + X_test.shape[0] == df_limpio.shape[0]:
    print("\n✅ La división es consistente con el total de la muestra.")

--- Dimensiones del Dataset ---
Muestra original (limpia): (4656, 18)
------------------------------
X_train (Entrenamiento):    (3724, 21)
y_train (Target Entrenam.): (3724,)
------------------------------
X_test  (Prueba):           (932, 21)
y_test  (Target Prueba):    (932,)

✅ La división es consistente con el total de la muestra.




*   Escalamiento de datos



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report)

# 1. Preparación: Eliminar columnas de texto que causan el ValueError
# Basado en tu cuaderno, esto quitará 'strat_col' y cualquier otra no numérica
X_train_final = X_train.select_dtypes(exclude=['object'])
X_test_final = X_test.select_dtypes(exclude=['object'])

# 2. Escalado (Paso crítico para que la Regresión Logística no de error de convergencia)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)



*   XGBoost



In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score

# 1. Configuración del modelo base
# tree_method='hist' es necesario para usar enable_categorical de forma eficiente
xgb_base = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    enable_categorical=True,
    tree_method='hist'
)

# 2. Definición de la rejilla de parámetros
param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8]
}

# 3. Configuración del GridSearchCV
grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# 4. Entrenamiento
# Usamos X_train_final que contiene las columnas numéricas y la categórica
grid_xgb.fit(X_train_scaled, y_train)

# 5. Evaluación con el conjunto de prueba
best_xgb = grid_xgb.best_estimator_
y_probs_xgb = best_xgb.predict_proba(X_test_final)[:, 1]
auc_final = roc_auc_score(y_test, y_probs_xgb)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:14:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

def guardar_artefactos_proyecto():
    # 1. Definir y asegurar que existan las rutas en tu Google Drive
    ruta_base = '/content/drive/MyDrive/SP1668 Tecnicas computacionales ML/Proyecto 3/'
    ruta_artefactos = os.path.join(ruta_base, 'Artefactos/')

    if not os.path.exists(ruta_artefactos):
        os.makedirs(ruta_artefactos)
        print(f"Creada la carpeta de artefactos en: {ruta_artefactos}")

    # ==========================================
    # 5. GUARDAR LOS ARTEFACTOS (.joblib)
    # ==========================================
    print("\nGuardando modelos, escaladores y codificadores...")

    # Guardar el modelo ganador XGBoost
    joblib.dump(best_xgb, os.path.join(ruta_artefactos, "best_xgb_model.joblib"))

    # Guardar el Escalador Estándar
    joblib.dump(scaler, os.path.join(ruta_artefactos, "scaler.joblib"))

    # Guardar el LabelEncoder de la variable ordinal de ingresos
    joblib.dump(le_income, os.path.join(ruta_artefactos, "label_encoder_income.joblib"))

    # Guardar el OneHotEncoder para las variables nominales
    joblib.dump(ohe, os.path.join(ruta_artefactos, "one_hot_encoder.joblib"))

    # Guardar la estructura exacta de las columnas finales del modelo
    columnas_modelo = X_train_final.columns.tolist()
    joblib.dump(columnas_modelo, os.path.join(ruta_artefactos, "columnas_modelo.joblib"))

    # ==========================================
    # 6. COMBINAR Y GUARDAR DATASETS A UN CSV
    # ==========================================
    print("Reconstruyendo y guardando dataset consolidado a CSV...")

    # --- Reconstrucción del set de ENTRENAMIENTO ---
    df_train = X_train_final.copy()

    # Añadimos las versiones escaladas correspondientes para auditoría posterior
    for i, name in enumerate(X_train_final.columns):
        df_train[f"{name}_scaled"] = X_train_scaled[:, i]

    df_train["target"] = y_train.values
    df_train["data_split"] = "train"  # <-- Corregido aquí

    # --- Reconstrucción del set de PRUEBA ---
    df_test = X_test_final.copy()

    for i, name in enumerate(X_test_final.columns):
        df_test[f"{name}_scaled"] = X_test_scaled[:, i]

    df_test["target"] = y_test.values
    df_test["data_split"] = "test"    # <-- Corregido aquí

    # --- Concatenación final ---
    full_dataset = pd.concat([df_train, df_test], ignore_index=True)

    csv_filename = os.path.join(ruta_base, "training_testing_data.csv")
    full_dataset.to_csv(csv_filename, index=False)

    # ==========================================
    # MENSAJES DE CONTROL (Auditoría Académica)
    # ==========================================
    print(f"\n¡Éxito! Se han exportado todos los recursos correctamente.")
    print(f"Ruta de artefactos: '{ruta_artefactos}'")
    print(f"1. best_xgb_model.joblib       -> Modelo óptimo seleccionado (XGBoost).")
    print(f"2. scaler.joblib               -> Instancia ajustada de StandardScaler (21 variables).")
    print(f"3. label_encoder_income.joblib -> Codificador ordinal para nivel de ingresos.")
    print(f"4. one_hot_encoder.joblib      -> Codificador Scikit-Learn para variables nominales.")
    print(f"5. columnas_modelo.joblib      -> Lista oficial con el orden exacto de las columnas.")
    print(f"6. Archivo CSV guardado en: '{csv_filename}'")

# Ejecución del guardado en el cuaderno
guardar_artefactos_proyecto()

Creada la carpeta de artefactos en: /content/drive/MyDrive/SP1668 Tecnicas computacionales ML/Proyecto 3/Artefactos/

Guardando modelos, escaladores y codificadores...
Reconstruyendo y guardando dataset consolidado a CSV...

¡Éxito! Se han exportado todos los recursos correctamente.
Ruta de artefactos: '/content/drive/MyDrive/SP1668 Tecnicas computacionales ML/Proyecto 3/Artefactos/'
1. best_xgb_model.joblib       -> Modelo óptimo seleccionado (XGBoost).
2. scaler.joblib               -> Instancia ajustada de StandardScaler (21 variables).
3. label_encoder_income.joblib -> Codificador ordinal para nivel de ingresos.
4. one_hot_encoder.joblib      -> Codificador Scikit-Learn para variables nominales.
5. columnas_modelo.joblib      -> Lista oficial con el orden exacto de las columnas.
6. Archivo CSV guardado en: '/content/drive/MyDrive/SP1668 Tecnicas computacionales ML/Proyecto 3/training_testing_data.csv'


In [ ]:
# Muestra la lista limpia de los nombres de las 21 columnas
print(X_train_final.columns.tolist())

['age', 'income_level', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'heart_rate', 'cholesterol_total', 'triglycerides', 'insulin_level', 'gender_Male', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Other', 'ethnicity_White', 'family_history_diabetes_1', 'hypertension_history_1', 'cardiovascular_history_1', 'smoking_status_Former', 'smoking_status_Never']
